In [ ]:
%load_ext mypy_ipython

In [ ]:
from langgraph.graph import START, END, StateGraph
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.runnables import Runnable
from collections.abc import Sequence
from typing import Literal

In [ ]:
class State (TypedDict):
    messages: Sequence[BaseMessage]

In [ ]:
state = State(messages = [HumanMessage("Could you tell me what is better an Audi a6 or BMW 5 series?")])

In [ ]:
state

In [ ]:
state['messages'][0].content

In [ ]:
chat = ChatOllama( model= 'llama3.2',seed=365, temperature=0, num_predict=256)

In [ ]:
response = chat.invoke(state['messages'])

In [ ]:
response

In [ ]:
def ask_question(state: State) -> State:
     print(f"\n --------> ENTERING ask_question:")
     print("What is your question?")
     return State(messages = [HumanMessage(input())])

In [ ]:
def ask_another_question(state: State) -> State:
     print(f"\n --------> ENTERING ask_another_question:")
     print("Would you like to ask one more question (yes/no)?")
     return State(messages = [HumanMessage(input())])

In [ ]:
ask_question(State(messages = []))

In [ ]:
def chatbot(state: State) -> State:

    print(f"\n --------> ENTERING chatbot:")
    
    response = chat.invoke(state['messages'])
    response.pretty_print()
    return State(messages = [response])

In [ ]:
chatbot(state)

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node('chatbot', chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge('chatbot', END)

In [ ]:
graph_compiled = graph.compile()

In [ ]:
isinstance(graph_compiled, Runnable)

In [ ]:
isinstance(graph, Runnable)

In [ ]:
graph_compiled.invoke(state)

In [ ]:
%mypy

# Conditional Edges and Routings

## Define the Routing Function

In [ ]:
def routing_function(state: State) -> Literal['ask_question', "__end__"]:
    if state['messages'][0].content == 'yes':
        return 'ask_question'
    else:
        return "__end__"

### Define the new Graph with Edges

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node('ask_question', ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node('ask_another_question', ask_another_question)


graph.add_edge(START, "ask_question")
graph.add_edge('ask_question', "chatbot")
graph.add_edge('chatbot', 'ask_another_question')
graph.add_conditional_edges(source='ask_another_question', path=routing_function)

In [ ]:
graph_compiled = graph.compile()

In [ ]:
graph_compiled

#### Test the Graph

In [ ]:
graph_compiled.invoke(State(messages = []))

In [ ]:
%mypy